In the previous notebook 1modeling_experiments.ipynb, we found that the Prophet model performed well in terms of forecasting rat sightings by day citywide. In this notebook, we will do some more feature engineering and hyperparameter tuning to obtain a better optimal model.

Because we wish this to be reusable, we will write things for Prophet and NeuralProphet separately. 

# Import Packages

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging


Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


# Prophet

## Import the data

In [2]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2025-02-28.
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to BROOKLYN

rs = rs[rs['borough']=='BROOKLYN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


In [3]:
rs["ds"] = pd.to_datetime(rs["ds"])
full_range = pd.date_range(start="2020-01-01", end="2025-02-28", freq="D")
# reindex and fill missing with 0
rs = rs.set_index("ds").reindex(full_range, fill_value=0).rename_axis("ds").reset_index()
rs

,ds,y
0,2020-01-01,7
1,2020-01-02,21
2,2020-01-03,13
3,2020-01-04,8
4,2020-01-05,5
...,...,...
1881,2025-02-24,20
1882,2025-02-25,24
1883,2025-02-26,28
1884,2025-02-27,31


## Prepare Prophet

In [4]:
date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [5]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [6]:
rs_saved = rs.copy()
df = rs.copy()

## Gridsearch for Hyperparameter Tuning for Prophet

In [7]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

In [8]:
# This code block is a grid search for hyperparameters.
# To tune for hyperparameters, add more possible parameters to the dictionary below and add more values to it.
# So far, the I've been able to get is {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5}

init_days = f'{len(rs)-(14*14)} days'
cv_period = '14 days'
forecast_horizon = '14 days'

param_grid = {  
    'changepoint_prior_scale': [0.1, 1],
    'seasonality_prior_scale': [1,2,5],
}

# Generate all combinations of parameters
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
rmses = []  # Store the RMSEs for each params here
performance = []

# Use cross validation to evaluate all parameters
for params in all_params:
    params['holidays'] = holidays
    m = Prophet(**params).fit(df)  # Fit model with given params
    df_cv = cross_validation(m, initial = init_days, period=cv_period, horizon = forecast_horizon)
    df_p = performance_metrics(df_cv, rolling_window=14)
    performance.append(df_p)
    rmses.append(df_p['rmse'].values[0])

# Find the best parameters
tuning_results = pd.DataFrame(all_params)
tuning_results['rmse'] = rmses

best_params = all_params[np.argmin(rmses)]

print(best_params)

14:39:00 - cmdstanpy - INFO - Chain [1] start processing
14:39:00 - cmdstanpy - INFO - Chain [1] done processing


  0%|          | 0/13 [00:00<?, ?it/s]

14:39:00 - cmdstanpy - INFO - Chain [1] start processing
14:39:00 - cmdstanpy - INFO - Chain [1] done processing
14:39:01 - cmdstanpy - INFO - Chain [1] start processing
14:39:01 - cmdstanpy - INFO - Chain [1] done processing
14:39:01 - cmdstanpy - INFO - Chain [1] start processing
14:39:01 - cmdstanpy - INFO - Chain [1] done processing
14:39:02 - cmdstanpy - INFO - Chain [1] start processing
14:39:02 - cmdstanpy - INFO - Chain [1] done processing
14:39:02 - cmdstanpy - INFO - Chain [1] start processing
14:39:02 - cmdstanpy - INFO - Chain [1] done processing
14:39:03 - cmdstanpy - INFO - Chain [1] start processing
14:39:03 - cmdstanpy - INFO - Chain [1] done processing
14:39:03 - cmdstanpy - INFO - Chain [1] start processing
14:39:03 - cmdstanpy - INFO - Chain [1] done processing
14:39:03 - cmdstanpy - INFO - Chain [1] start processing
14:39:04 - cmdstanpy - INFO - Chain [1] done processing
14:39:04 - cmdstanpy - INFO - Chain [1] start processing
14:39:04 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/13 [00:00<?, ?it/s]

14:39:07 - cmdstanpy - INFO - Chain [1] start processing
14:39:07 - cmdstanpy - INFO - Chain [1] done processing
14:39:07 - cmdstanpy - INFO - Chain [1] start processing
14:39:07 - cmdstanpy - INFO - Chain [1] done processing
14:39:08 - cmdstanpy - INFO - Chain [1] start processing
14:39:08 - cmdstanpy - INFO - Chain [1] done processing
14:39:08 - cmdstanpy - INFO - Chain [1] start processing
14:39:08 - cmdstanpy - INFO - Chain [1] done processing
14:39:08 - cmdstanpy - INFO - Chain [1] start processing
14:39:09 - cmdstanpy - INFO - Chain [1] done processing
14:39:09 - cmdstanpy - INFO - Chain [1] start processing
14:39:09 - cmdstanpy - INFO - Chain [1] done processing
14:39:09 - cmdstanpy - INFO - Chain [1] start processing
14:39:09 - cmdstanpy - INFO - Chain [1] done processing
14:39:10 - cmdstanpy - INFO - Chain [1] start processing
14:39:10 - cmdstanpy - INFO - Chain [1] done processing
14:39:10 - cmdstanpy - INFO - Chain [1] start processing
14:39:10 - cmdstanpy - INFO - Chain [1]

...

  0%|          | 0/13 [00:00<?, ?it/s]

14:39:30 - cmdstanpy - INFO - Chain [1] start processing
14:39:31 - cmdstanpy - INFO - Chain [1] done processing
14:39:31 - cmdstanpy - INFO - Chain [1] start processing
14:39:32 - cmdstanpy - INFO - Chain [1] done processing
14:39:32 - cmdstanpy - INFO - Chain [1] start processing
14:39:33 - cmdstanpy - INFO - Chain [1] done processing
14:39:33 - cmdstanpy - INFO - Chain [1] start processing
14:39:33 - cmdstanpy - INFO - Chain [1] done processing
14:39:34 - cmdstanpy - INFO - Chain [1] start processing
14:39:34 - cmdstanpy - INFO - Chain [1] done processing
14:39:35 - cmdstanpy - INFO - Chain [1] start processing
14:39:35 - cmdstanpy - INFO - Chain [1] done processing
14:39:35 - cmdstanpy - INFO - Chain [1] start processing
14:39:36 - cmdstanpy - INFO - Chain [1] done processing
14:39:36 - cmdstanpy - INFO - Chain [1] start processing
14:39:37 - cmdstanpy - INFO - Chain [1] done processing
14:39:37 - cmdstanpy - INFO - Chain [1] start processing
14:39:37 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/13 [00:00<?, ?it/s]

14:39:42 - cmdstanpy - INFO - Chain [1] start processing
14:39:42 - cmdstanpy - INFO - Chain [1] done processing
14:39:42 - cmdstanpy - INFO - Chain [1] start processing
14:39:43 - cmdstanpy - INFO - Chain [1] done processing
14:39:43 - cmdstanpy - INFO - Chain [1] start processing
14:39:43 - cmdstanpy - INFO - Chain [1] done processing
14:39:44 - cmdstanpy - INFO - Chain [1] start processing
14:39:44 - cmdstanpy - INFO - Chain [1] done processing
14:39:44 - cmdstanpy - INFO - Chain [1] start processing
14:39:45 - cmdstanpy - INFO - Chain [1] done processing
14:39:45 - cmdstanpy - INFO - Chain [1] start processing
14:39:46 - cmdstanpy - INFO - Chain [1] done processing
14:39:46 - cmdstanpy - INFO - Chain [1] start processing
14:39:46 - cmdstanpy - INFO - Chain [1] done processing
14:39:47 - cmdstanpy - INFO - Chain [1] start processing
14:39:47 - cmdstanpy - INFO - Chain [1] done processing
14:39:48 - cmdstanpy - INFO - Chain [1] start processing
14:39:48 - cmdstanpy - INFO - Chain [1]

{'changepoint_prior_scale': 1, 'seasonality_prior_scale': 2, 'holidays':        holiday         ds  lower_window  upper_window
0   federal_us 2020-01-01             0             1
1   federal_us 2020-01-20             0             1
2   federal_us 2020-02-17             0             1
3   federal_us 2020-05-25             0             1
4   federal_us 2020-07-03             0             1
5   federal_us 2020-09-07             0             1
6   federal_us 2020-10-12             0             1
7   federal_us 2020-11-11             0             1
8   federal_us 2020-11-26             0             1
9   federal_us 2020-12-25             0             1
10  federal_us 2021-01-01             0             1
11  federal_us 2021-01-18             0             1
12  federal_us 2021-02-15             0             1
13  federal_us 2021-05-31             0             1
14  federal_us 2021-06-18             0             1
15  federal_us 2021-07-05             0             1
16  feder

In [9]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(**best_params)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    y_pred = np.round(y_pred)
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Calculate MAPE
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# Convert results to a datafrane
prophet_results_df = pd.DataFrame(results)

14:39:52 - cmdstanpy - INFO - Chain [1] start processing
14:39:52 - cmdstanpy - INFO - Chain [1] done processing
14:39:53 - cmdstanpy - INFO - Chain [1] start processing
14:39:53 - cmdstanpy - INFO - Chain [1] done processing
14:39:54 - cmdstanpy - INFO - Chain [1] start processing
14:39:54 - cmdstanpy - INFO - Chain [1] done processing
14:39:55 - cmdstanpy - INFO - Chain [1] start processing
14:39:55 - cmdstanpy - INFO - Chain [1] done processing
14:39:56 - cmdstanpy - INFO - Chain [1] start processing
14:39:56 - cmdstanpy - INFO - Chain [1] done processing
14:39:57 - cmdstanpy - INFO - Chain [1] start processing
14:39:57 - cmdstanpy - INFO - Chain [1] done processing
14:39:58 - cmdstanpy - INFO - Chain [1] start processing
14:39:58 - cmdstanpy - INFO - Chain [1] done processing
14:39:59 - cmdstanpy - INFO - Chain [1] start processing
14:39:59 - cmdstanpy - INFO - Chain [1] done processing
14:40:00 - cmdstanpy - INFO - Chain [1] start processing
14:40:00 - cmdstanpy - INFO - Chain [1]

## Prophet's Results

In [10]:
prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]
prophet_results_df

,fold,rmse,mape
0,0,7.275203,0.275694
1,1,11.038892,0.596755
2,2,8.040078,0.358834
3,3,4.511889,0.099649
4,4,7.333874,0.122950
5,5,10.049876,0.211894
6,6,7.396911,0.181441
7,7,8.128433,0.259616
8,8,8.228869,0.285086
9,9,6.059231,0.164031


It is possible that the average RMSE displayed above is worse than the one found for Prophet in 1modeling_experiments.ipynb. This would just mean that the gridsearch did not beat the default settings.

In [11]:
np.NaN = np.nan


# the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# the errors that do show up are not all that important and a lot is due to outdated packages.
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("neuralprophet").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("NP").setLevel(logging.ERROR)

## Import the data

In [12]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to BROOKLYN

rs = rs[rs['borough']=='BROOKLYN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

rs.head()

,ds,y
1,2020-01-01,7
6,2020-01-02,21
11,2020-01-03,13
16,2020-01-04,8
20,2020-01-05,5


In [13]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [14]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)


regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

Uncomment the code block below to run Optuna's hyperparameter tuning. Make sure to then comment away the codeblock following that which has a saved result from a previous run.

In [15]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=2, test_size=14)


def objective(trial):
    regressor_lags = {
        'apparent_temperature_max': trial.suggest_int('lag_temp_max', 1, 60),
        'apparent_temperature_min': trial.suggest_int('lag_temp_min', 1, 60),
        'snowfall_sum': trial.suggest_int('lag_snowfall', 1, 7),
    }
    n_lags = trial.suggest_int('n_lags', 1, 60)
    epochs = trial.suggest_int('epochs', 10, 250)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 1, log=True)
    batch_size = trial.suggest_int('batch_size', 12, 248)
    ar_reg = trial.suggest_float('ar_reg', 0.5, 3)
    fold_rmses = []
    for i, (train_idx, test_idx) in enumerate(tscv.split(rs)):

        train = rs.iloc[train_idx].copy()
        test = rs.iloc[test_idx].copy()
        
        existing_regressors = [col for col in regressed_features if col in train.columns]
        train = train.dropna(subset=["y"] + existing_regressors)
        test = test.dropna(subset=existing_regressors)
        
        # Skip fold if too few rows
        if len(train) < 20 or len(test) < 1:
            continue
        
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            n_lags=n_lags,
            epochs=epochs,
            ar_reg = ar_reg,
            accelerator="auto",   # uses GPU if available
            learning_rate=learning_rate,
            batch_size=batch_size
        )
        model.add_country_holidays(country_name="US")
        for col in existing_regressors:
            model.add_lagged_regressor(col, n_lags=regressor_lags[col])
        
        model.fit(train, freq="D", progress="off")
        future = pd.concat([
            train[['ds','y'] + existing_regressors],
            test[['ds','y']].merge(wd[['ds'] + existing_regressors], on="ds", how="left")
        ])
        future = future.dropna(subset=existing_regressors)
        forecast = model.predict(future)
        
        y_pred = forecast["yhat1"].iloc[-len(test):].values
        y_true = test["y"].values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(rmse)

        intermediate_score = np.mean(fold_rmses)
        trial.report(intermediate_score, i)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
    return np.mean(fold_rmses) if fold_rmses else float("inf")

study = optuna.create_study(direction="minimize")
study = optuna.create_study(
    direction="minimize",
    study_name="model_neural",
    storage="sqlite:///model_neural.db",
    load_if_exists=True
)
study.optimize(objective, n_trials=5)  # adjust n_trials as needed



best_params = study.best_params

print("Best Parameters", best_params)
print("Best RMSE:", study.best_value)

[I 2026-03-14 14:40:20,693] A new study created in memory with name: no-name-88ea385f-878e-4fe0-9fd5-890285f2545c
[I 2026-03-14 14:40:21,492] Using an existing study with name 'model_neural' instead of creating a new one.


Training: 0it [00:00, ?it/s]

Predicting: 19it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 19it [00:00, ?it/s]

...

[I 2026-03-14 14:42:52,721] Trial 10 finished with value: 4.624638013541631 and parameters: {'lag_temp_max': 57, 'lag_temp_min': 5, 'lag_snowfall': 7, 'n_lags': 59, 'epochs': 72, 'learning_rate': 0.027222201288698115, 'batch_size': 53, 'ar_reg': 0.8089311255395453}. Best is trial 10 with value: 4.624638013541631.


Training: 0it [00:00, ?it/s]

Predicting: 27it [00:00, ?it/s]

[I 2026-03-14 14:42:58,125] Trial 11 pruned. 


Best Parameters {'lag_temp_max': 57, 'lag_temp_min': 5, 'lag_snowfall': 7, 'n_lags': 59, 'epochs': 72, 'learning_rate': 0.027222201288698115, 'batch_size': 53, 'ar_reg': 0.8089311255395453}
Best RMSE: 4.624638013541631


In [16]:
# best_params = dict({'lag_temp_max': 54, 'lag_temp_min': 18, 
#                     'lag_snowfall': 1, 'n_lags': 59, 'epochs': 493, 'learning_rate': 0.003214767890388168, 'batch_size': 220, 'ar_reg': 0.5847571241076923})

In [17]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to BROOKLYN

rs = rs[rs['borough']=='BROOKLYN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

wd = pd.DataFrame(data["daily"])
wd["date"] = pd.to_datetime(wd["time"])
wd = wd.set_index("date")

## Evaluate the Model

In [18]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min','snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']


results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    train = train.dropna(subset=["y"])

    test = rs.iloc[test_index].copy()


    model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          learning_rate = best_params['learning_rate'],
                          epochs = best_params['epochs'],
                          n_lags= best_params['n_lags'],
                          ar_reg=best_params['ar_reg'],
                          accelerator="auto",   # uses GPU if available
                          batch_size= best_params['batch_size']
                          )
    model = model.add_country_holidays(country_name="US")
    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])
        
    model.fit(train, freq="D", progress="off")

    # build dataframe containing future regressors
    future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_true = test["y"].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse, "mape": mape})

neural_prophet_results_df = pd.DataFrame(results)
neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]
neural_prophet_results_df

Training: 0it [00:00, ?it/s]

Predicting: 35it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 35it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

...

Training: 0it [00:00, ?it/s]

Predicting: 41it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 42it [00:00, ?it/s]

,fold,rmse,mape
0,0,7.383290,0.349025
1,1,4.182768,0.147146
2,2,5.077433,0.209848
3,3,5.979543,0.158442
4,4,7.296470,0.303452
5,5,5.572516,0.196689
6,6,6.735408,0.210421
7,7,7.206034,0.336554
8,8,5.715506,0.206138
9,9,6.550706,0.200571


# Neural Prophet

## Load and Prepare Data

In [3]:
np.NaN = np.nan


# the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# the errors that do show up are not all that important and a lot is due to outdated packages.
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("neuralprophet").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("NP").setLevel(logging.ERROR)

In [4]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data 
rs = rs[rs['created_date']<'2025-02-28']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to BROOKLYN

rs = rs[rs['borough']=='BROOKLYN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

rs

,ds,y
1,2020-01-01,7
6,2020-01-02,21
11,2020-01-03,13
16,2020-01-04,8
20,2020-01-05,5
...,...,...
8899,2025-02-23,5
8904,2025-02-24,20
8909,2025-02-25,24
8914,2025-02-26,28


In [5]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28" 

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("../../scr/data/weather_data/wd_2020_to_march_2025.csv")
    nd = nd.set_index('time')
    wd = nd
    wd["date"] = pd.to_datetime(wd["time"])
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [6]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)


regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

## Optuna Hyperparameter Tuning

In [ ]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


def objective(trial):
    regressor_lags = {
        'apparent_temperature_max': trial.suggest_int('lag_temp_max', 1, 90),
        'apparent_temperature_min': trial.suggest_int('lag_temp_min', 1, 90),
        'snowfall_sum': trial.suggest_int('lag_snowfall', 1, 14),
    }
    n_lags = trial.suggest_int('n_lags', 1, 90)
    epochs = trial.suggest_int('epochs', 10, 250)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 1, log=True)
    batch_size = trial.suggest_int('batch_size', 1024, 2048)
    ar_reg = trial.suggest_float('ar_reg', 0.5, 3)
    fold_rmses = []
    for i, (train_idx, test_idx) in enumerate(tscv.split(rs)):

        train = rs.iloc[train_idx].copy()
        test = rs.iloc[test_idx].copy()
        
        existing_regressors = [col for col in regressed_features if col in train.columns]
        train = train.dropna(subset=["y"] + existing_regressors)
        test = test.dropna(subset=existing_regressors)
        
        # Skip fold if too few rows
        if len(train) < 20 or len(test) < 1:
            continue
        
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            n_lags=n_lags,
            epochs=epochs,
            ar_reg = ar_reg,
            accelerator="auto",   # uses GPU if available
            learning_rate=learning_rate,
            batch_size=batch_size
        )
        model.add_country_holidays(country_name="US")
        for col in existing_regressors:
            model.add_lagged_regressor(col, n_lags=regressor_lags[col])
        
        model.fit(train, freq="D", progress="off")
        future = pd.concat([
            train[['ds','y'] + existing_regressors],
            test[['ds','y']].merge(wd[['ds'] + existing_regressors], on="ds", how="left")
        ])
        future = future.dropna(subset=existing_regressors)
        forecast = model.predict(future)
        
        y_pred = forecast["yhat1"].iloc[-len(test):].values
        y_pred = np.round(y_pred)
        y_true = test["y"].values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(rmse)
        
    return np.mean(fold_rmses)

study = optuna.create_study(
    direction="minimize",
    study_name="model_neural",
    storage="sqlite:///model_neural.db",
    load_if_exists=True
)
study.optimize(objective, n_trials=100)  # adjust n_trials as needed



best_params = study.best_params

print("Best Parameters", best_params)
print("Best RMSE:", study.best_value)

[I 2026-03-15 08:14:12,998] Using an existing study with name 'model_neural' instead of creating a new one.


Training: 0it [00:00, ?it/s]

Predicting: 2it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 2it [00:00, ?it/s]

...

Predicting: 2it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 2it [00:00, ?it/s]

## Evaluate the Model's Performance

In [ ]:
# best_params = dict({'lag_temp_max': 54, 'lag_temp_min': 18, 
#                     'lag_snowfall': 1, 'n_lags': 59, 'epochs': 493, 'learning_rate': 0.003214767890388168, 'batch_size': 220, 'ar_reg': 0.5847571241076923})

In [ ]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to BROOKLYN

rs = rs[rs['borough']=='BROOKLYN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


## Add weather data.
import requests


lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()


if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [ ]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min','snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']


results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    train = train.dropna(subset=["y"])

    test = rs.iloc[test_index].copy()


    model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          learning_rate = best_params['learning_rate'],
                          epochs = best_params['epochs'],
                          n_lags= best_params['n_lags'],
                          ar_reg=best_params['ar_reg'],
                          accelerator="auto",   # uses GPU if available
                          batch_size= best_params['batch_size']
                          )
    model = model.add_country_holidays(country_name="US")
    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])
        
    model.fit(train, freq="D", progress="off")

    # build dataframe containing future regressors
    future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_pred = np.round(y_pred)
    y_true = test["y"].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse, "mape": mape})

neural_prophet_results_df = pd.DataFrame(results)
neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]

Training: 0it [00:00, ?it/s]

Predicting: 28it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 28it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

...

Predicting: 34it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 34it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 35it [00:00, ?it/s]

## Neural Prophet's Results

In [ ]:
neural_prophet_results_df

,fold,rmse,mape
0,0,6.546537,0.251424
1,1,6.766514,0.343668
2,2,7.166390,0.296326
3,3,5.994045,0.138407
4,4,7.941752,0.169197
5,5,9.296697,0.194414
6,6,8.831761,0.193492
7,7,10.433462,0.301299
8,8,7.372730,0.238395
9,9,6.313251,0.151453
